In [2]:
import os
from dotenv import load_dotenv
from typing import TypedDict, List
import operator
import nest_asyncio

# Apply patch for running async in Jupyter
nest_asyncio.apply()

# Core LangGraph components
from langgraph.graph import StateGraph, START, END
# FIX: Import RunnableLambda from langchain_core
from langchain_core.runnables import RunnableLambda

# LLM and messages
from langchain_openai import ChatOpenAI
from langchain_core.messages import HumanMessage

# Load API keys and set up tracing
load_dotenv()
os.environ["LANGCHAIN_TRACING_V2"] = "true"
os.environ["LANGSMITH_PROJECT"] = "Intro to LangGraph"

# Define the State
class GraphState(TypedDict):
    question: str
    expert_1_answer: str
    expert_2_answer: str

# Define Nodes
def expert_1(state: GraphState):
    # This expert answers concisely
    print("--- Expert 1 Answering ---")
    llm = ChatOpenAI(model="gpt-4o", temperature=0)
    prompt = f"Answer the following question very concisely: {state['question']}"
    response = llm.invoke(prompt)
    return {"expert_1_answer": response.content}

def expert_2(state: GraphState):
    # This expert answers with more detail
    print("--- Expert 2 Answering ---")
    llm = ChatOpenAI(model="gpt-4o", temperature=0.7) # Higher temp for variety
    prompt = f"Answer the following question with some detail: {state['question']}"
    response = llm.invoke(prompt)
    return {"expert_2_answer": response.content}

# Build the Graph
workflow = StateGraph(GraphState)

# Add the nodes that will run in parallel
# Wrap them in RunnableLambda to ensure they run concurrently
workflow.add_node("expert1", RunnableLambda(expert_1))
workflow.add_node("expert2", RunnableLambda(expert_2))

# The START node leads to *both* expert nodes simultaneously
workflow.add_edge(START, "expert1")
workflow.add_edge(START, "expert2")

# Both expert nodes must finish before the graph can end
# LangGraph waits for parallel branches automatically before proceeding
workflow.add_edge("expert1", END)
workflow.add_edge("expert2", END)

# Compile the graph
app = workflow.compile()

# --- Run the Graph ---
print("--- Running Parallel Experts ---")
inputs = {"question": "What is the difference between LangGraph and LCEL?"}
final_state = app.invoke(inputs)

print("\n--- Results ---")
print(f"Expert 1 (Concise): {final_state['expert_1_answer']}")
print(f"Expert 2 (Detailed): {final_state['expert_2_answer']}")

--- Running Parallel Experts ---
--- Expert 1 Answering ---
--- Expert 2 Answering ---

--- Results ---
Expert 1 (Concise): LangGraph and LCEL are both tools related to language models, but they serve different purposes. LangGraph is a framework designed for building and deploying language models with a focus on graph-based representations and interactions. LCEL, on the other hand, stands for Language-Conditioned Embodied Learning, which involves training agents to perform tasks in simulated environments using language instructions. The key difference lies in their applications: LangGraph is more about structuring and deploying language models, while LCEL is about integrating language understanding with physical task execution.
Expert 2 (Detailed): LangGraph and LCEL (Language and Communication Expressiveness Layer) are both concepts related to language processing and computational linguistics, but they serve different purposes and are utilized in distinct contexts. Here's a detailed b